In [1]:
import pandas as pd
import requests
from io import BytesIO

# NHANES 2017-2018 cycle files we need
files = {
    "demographics": "DEMO_J",
    "diabetes": "DIQ_J",
    "depression": "DPQ_J",
    "hba1c": "GHB_J",
    "utilization": "HUQ_J"
}

base_url = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/{}.xpt"
headers = {"User-Agent": "Mozilla/5.0"}  # some CDC endpoints block requests with no user-agent

data = {}

for name, code in files.items():
    url = base_url.format(code)
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    df = pd.read_sas(BytesIO(response.content), format="xport")
    data[name] = df
    print(f"{name} ({code}): {df.shape[0]} rows, {df.shape[1]} columns")

demographics (DEMO_J): 9254 rows, 46 columns
diabetes (DIQ_J): 8897 rows, 54 columns
depression (DPQ_J): 5533 rows, 11 columns
hba1c (GHB_J): 6401 rows, 2 columns
utilization (HUQ_J): 9254 rows, 10 columns


In [2]:
import os

os.makedirs("nhanes_raw", exist_ok=True)

for name, df in data.items():
    df.to_csv(f"nhanes_raw/{name}.csv", index=False)

print("All files saved.")

All files saved.


In [3]:
url = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/TRIGLY_J.xpt"
headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(url, headers=headers)
response.raise_for_status()
df_ldl = pd.read_sas(BytesIO(response.content), format="xport")
data["ldl"] = df_ldl

df_ldl.to_csv("nhanes_raw/ldl.csv", index=False)

print(f"ldl (TRIGLY_J): {df_ldl.shape[0]} rows, {df_ldl.shape[1]} columns")

ldl (TRIGLY_J): 3036 rows, 10 columns


In [4]:
import shutil
from google.colab import files

shutil.make_archive("ldl_export", "zip", "nhanes_raw")
files.download("ldl_export.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>